# Heavy Vehicle Failure Prediction Through Intelligent Sensor Data Analysis

## Exploratory Data Analysis and Data Preprocessing — APS Scania Dataset

**Dataset:** APS Failure at Scania Trucks  
**Target:** `class`  
**Task:** Binary classification of APS-related failure (`pos`) vs. non-APS-related failure (`neg`)

> This notebook intentionally starts from the combined raw dataset. Preprocessing decisions are made from EDA results and are fitted using training data only where appropriate to avoid data leakage.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")


## 2. Load Dataset

In [ ]:
# Update this path if your CSV is stored somewhere else
DATA_PATH = "APS_Scania_Complete_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


## 3. Initial Dataset Inspection

In [ ]:
display(df.head())
display(df.tail())

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


In [ ]:
print("Column names:")
print(df.columns.tolist())


In [ ]:
df.info()


## 4. Target Variable Analysis

The `class` column is the prediction target:

- `neg` → failure is not related to the APS component
- `pos` → failure is related to the APS component


In [ ]:
class_counts = df["class"].value_counts(dropna=False)
class_percent = df["class"].value_counts(normalize=True, dropna=False) * 100

print("Class counts:")
display(class_counts)

print("Class percentages:")
display(class_percent.round(2))


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="class")
plt.title("APS Failure Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Records")
plt.show()


### EDA Observation

The target distribution should be checked for class imbalance. APS failure prediction is expected to be highly imbalanced, so accuracy alone should not be used to judge the final model.


## 5. Raw Missing-Value Analysis

In [ ]:
# In the original APS data, missing values are represented by the string "na".
missing_count_raw = (df == "na").sum()
missing_percentage_raw = (missing_count_raw / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count_raw,
    "Missing_Percentage": missing_percentage_raw
}).sort_values("Missing_Percentage", ascending=False)

display(missing_summary.head(30))


In [ ]:
top_missing = missing_summary.head(20)

plt.figure(figsize=(12, 7))
sns.barplot(
    x=top_missing["Missing_Percentage"],
    y=top_missing.index
)
plt.title("Top 20 Features by Missing-Value Percentage")
plt.xlabel("Missing Values (%)")
plt.ylabel("Feature")
plt.show()


## 6. Convert `na` to Missing Values and Convert Features to Numeric

This is a data-cleaning step. We do not impute values yet; first we inspect the cleaned data and decide the preprocessing strategy.


In [ ]:
df_clean = df.copy()

# Convert the dataset's "na" representation to actual missing values.
df_clean = df_clean.replace("na", np.nan)

# Convert all feature columns to numeric.
feature_cols = [c for c in df_clean.columns if c != "class"]
df_clean[feature_cols] = df_clean[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

# Encode the target.
df_clean["class"] = df_clean["class"].map({"neg": 0, "pos": 1})

print(df_clean.shape)
print(df_clean["class"].value_counts(dropna=False))


In [ ]:
missing_clean = df_clean.isna().sum()
missing_clean_pct = (missing_clean / len(df_clean)) * 100

missing_clean_summary = pd.DataFrame({
    "Missing_Count": missing_clean,
    "Missing_Percentage": missing_clean_pct
}).sort_values("Missing_Percentage", ascending=False)

display(missing_clean_summary.head(30))


## 7. Statistical Summary

In [ ]:
X_raw = df_clean.drop(columns="class")

display(X_raw.describe().T)


## 8. Low-Variance Feature Analysis

In [ ]:
feature_variance = X_raw.var(numeric_only=True).sort_values()

display(feature_variance.head(20))


Low-variance features are inspected rather than automatically deleted. A sensor feature with low variance may still be useful for a classifier, so removal should be justified.


## 9. Feature Distribution Analysis

In [ ]:
# Plot a small representative set first instead of plotting all 170 features.
candidate_features = [
    c for c in ["aa_000", "ag_000", "ay_000", "az_000", "ba_000"]
    if c in X_raw.columns
]

for col in candidate_features:
    plt.figure(figsize=(7, 5))
    sns.histplot(X_raw[col].dropna(), kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


## 10. Outlier Analysis

In [ ]:
for col in candidate_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=X_raw[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.show()


### Outlier Decision

Do not automatically remove every extreme value. In vehicle sensor data, an extreme measurement can represent a genuine abnormal operating condition that may be useful for failure prediction. We will therefore retain values unless the EDA identifies clear evidence of invalid measurements.


## 11. Correlation Analysis

In [ ]:
corr = X_raw.corr(numeric_only=True)

# Display correlations among the first 30 features for readability.
plt.figure(figsize=(14, 10))
sns.heatmap(corr.iloc[:30, :30], cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — First 30 Features")
plt.show()


In [ ]:
# Find highly correlated feature pairs.
threshold = 0.90
high_corr_pairs = []

cols = corr.columns
for i in range(len(cols)):
    for j in range(i):
        value = corr.iloc[i, j]
        if pd.notna(value) and abs(value) >= threshold:
            high_corr_pairs.append((cols[i], cols[j], value))

high_corr_pairs = sorted(
    high_corr_pairs,
    key=lambda x: abs(x[2]),
    reverse=True
)

print("Number of highly correlated pairs:", len(high_corr_pairs))
display(pd.DataFrame(
    high_corr_pairs[:30],
    columns=["Feature_1", "Feature_2", "Correlation"]
))


## 12. Train-Test Split

Because this combined file contains the target labels, we create our own stratified split for the project. The test set will remain untouched while preprocessing parameters are learned from the training set.


In [ ]:
X = df_clean.drop(columns="class")
y = df_clean["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining class distribution:")
display(y_train.value_counts())

print("\nTesting class distribution:")
display(y_test.value_counts())


## 13. Missing-Value Preprocessing

A median imputer is fitted **only on the training data**. The same learned medians are then applied to the test data. This avoids using information from the test set during training.


In [ ]:
# Check how many columns have missing values in the training data.
train_missing_pct = X_train.isna().mean() * 100

display(
    train_missing_pct.sort_values(ascending=False).head(30).to_frame("Training_Missing_%")
)


In [ ]:
# Initial, conservative approach:
# retain features and impute numerical missing values using the training median.
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Training data after imputation:", X_train_imputed.shape)
print("Testing data after imputation :", X_test_imputed.shape)


## 14. Feature Scaling

Scaling is model-dependent. It is useful for models such as Logistic Regression and SVM, while tree-based models generally do not require it.

The scaled matrices below are intended for scale-sensitive models.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape :", X_test_scaled.shape)


## 15. Class Imbalance Check After Split

In [ ]:
print("Training class distribution:")
print(y_train.value_counts())
print()

print("Training class percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))


### Imbalance Strategy

The positive APS-failure class is much smaller than the negative class. For the first model baseline, use class-weighted algorithms rather than applying SMOTE blindly. SMOTE can be evaluated later as a separate experiment if required.


## 16. Save Processed Data

In [ ]:
# Preserve feature names after preprocessing.
feature_names = X_train.columns.tolist()

X_train_processed_df = pd.DataFrame(
    X_train_imputed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_imputed,
    columns=feature_names,
    index=X_test.index
)

X_train_processed_df.to_csv("X_train_processed.csv", index=False)
X_test_processed_df.to_csv("X_test_processed.csv", index=False)

pd.Series(
    y_train.to_numpy(),
    name="class"
).to_csv("y_train.csv", index=False)

pd.Series(
    y_test.to_numpy(),
    name="class"
).to_csv("y_test.csv", index=False)

print("Processed APS Scania training and testing files saved successfully!")


## 17. Final Preprocessing Summary

### Completed
- Loaded the combined APS Scania dataset
- Inspected shape, columns and data types
- Analyzed target distribution
- Identified the `na` missing-value representation
- Converted missing values to `NaN`
- Converted sensor features to numeric
- Encoded `neg`/`pos` as 0/1
- Performed statistical, distribution, outlier and correlation analysis
- Created a stratified 80/20 train-test split
- Applied median imputation using training data only
- Prepared scaled data for scale-sensitive ML models
- Saved processed training/test feature and target files

### Important
Class-imbalance handling is intentionally kept as a model-development experiment rather than modifying the test data. Model training and evaluation should be done in the next notebook section.
